In [1]:
from pyspark.sql import functions as F

StatementMeta(, 6eda7859-93b0-44fb-b954-a9bc11c30c66, 3, Finished, Available, Finished, False)

In [2]:
# 1) Read updated positions
updated_positions = (
    spark.table("dbo.updated_positions")
    .withColumn("position_date", F.to_date("position_date"))
    .withColumn("quantity", F.col("quantity").cast("double"))
)

StatementMeta(, 6eda7859-93b0-44fb-b954-a9bc11c30c66, 4, Finished, Available, Finished, False)

In [3]:
# 2) Keep latest snapshot date
latest_position_date = updated_positions.select(F.max("position_date").alias("d")).first()["d"]

StatementMeta(, 6eda7859-93b0-44fb-b954-a9bc11c30c66, 5, Finished, Available, Finished, False)

In [4]:
# 3) Aggregate all trader positions by asset and calculate 25% limit
limit_positions = (
    updated_positions
    .filter(F.col("position_date") == F.lit(latest_position_date))
    .groupBy("ticker")
    .agg(F.sum("quantity").alias("total_position_qty"))
    .withColumn("position_limit_qty", F.col("total_position_qty") * F.lit(0.25))
    .withColumn("position_date", F.lit(latest_position_date))
    .select("position_date", "ticker", "total_position_qty", "position_limit_qty")
)

StatementMeta(, 6eda7859-93b0-44fb-b954-a9bc11c30c66, 6, Finished, Available, Finished, False)

In [5]:
# 4) Persist output
limit_positions.write.mode("overwrite").saveAsTable("dbo.limit_positions")

StatementMeta(, 6eda7859-93b0-44fb-b954-a9bc11c30c66, 7, Finished, Available, Finished, False)

In [6]:
display(limit_positions.orderBy("ticker"))

StatementMeta(, 6eda7859-93b0-44fb-b954-a9bc11c30c66, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 15746269-cb23-4a43-a5f1-5458cd1ab2c8)